## CASE — ¿se recupera la frecuencia respiratoria desde la PPG de dedo?
Paso de factibilidad del estudio de reducción de sensores.
- **Referencia:** banda respiratoria torácica (`rsp`, 1000 Hz): respiraciones detectadas en la señal.
  Solo se usan ventanas donde el conteo de respiraciones y la FFT de la banda coinciden (≤ 2 rpm).
- **Estimación:** PPG de dedo (`bvp`, 1000 Hz) con la FFT congelada de BIDMC/CapnoBase, sin cambios.
- Resultados por categoría de vídeo (divertido, aburrido, relajante, miedo).

In [ ]:
# k-1 — Imports, configuración congelada y rutas
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import butter, decimate, detrend, filtfilt, find_peaks, resample_poly

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CFG = json.loads((BASE_DIR / "results" / "config_congelada.json").read_text())
CASE_DIR = BASE_DIR / "data" / "CASE" / "data" / "interpolated" / "physiological"   # sub_1.csv … sub_30.csv
FS, FS_CASE = 125, 1000
VENTANA_S, PASO_S = CFG["ventana_s"], CFG["paso_s"]
RPM_MIN, RPM_MAX = CFG["rpm_min"], CFG["rpm_max"]
N = VENTANA_S * FS
CATEGORIA = {1: "divertido", 2: "divertido", 3: "aburrido", 4: "aburrido",
             5: "relajante", 6: "relajante", 7: "miedo", 8: "miedo"}    # 10, 11, 12 = inicio, pantalla azul, fin
ARCHIVOS = sorted(CASE_DIR.glob("sub_*.csv"), key=lambda p: int(p.stem.split("_")[1]))
print(len(ARCHIVOS), "participantes en", CASE_DIR)


In [ ]:
# k-2 — Funciones (mismas que en WESAD, con fs = 1000 Hz)
def leer(p):
    d = pd.read_csv(p, usecols=["daqtime", "bvp", "rsp", "video"])
    return d["bvp"].to_numpy(float), d["rsp"].to_numpy(float), d["video"].to_numpy(int)

def marcas_respiracion(rsp, fs=FS_CASE):
    x = decimate(decimate(rsp, 10, zero_phase=True), 4, zero_phase=True)      # 1000 -> 25 Hz
    f = fs / 40
    b, a = butter(2, [0.05, 1.0], btype="band", fs=f)
    x = filtfilt(b, a, x)
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    picos, _ = find_peaks(x, distance=int(1.0 * f), prominence=0.25 * iqr)
    return picos / f

def rr_ref(m, t0, t1, min_marcas=3):
    m = m[(m >= t0) & (m < t1)]
    return np.nan if len(m) < min_marcas else 60.0 / np.median(np.diff(m))

def rr_fft(seg, fs=FS, nfft=2048):
    q = int(fs // 5)
    x = decimate(decimate(seg, 5, zero_phase=True), q // 5, zero_phase=True)
    x = detrend(x) * np.hanning(len(x))
    f = np.fft.rfftfreq(nfft, q / fs) * 60
    P = np.abs(np.fft.rfft(x, nfft)) ** 2
    banda = (f >= RPM_MIN) & (f <= RPM_MAX)
    return f[np.argmax(np.where(banda, P, -np.inf))]

def evaluar(p):
    bvp, rsp, vid = leer(p)
    marcas = marcas_respiracion(rsp)
    ppg = resample_poly(bvp, FS, FS_CASE)
    rsp125 = resample_poly(rsp, FS, FS_CASE)
    b, a = butter(4, CFG["lp_fft_hz"], btype="low", fs=FS)
    riiv_fft = filtfilt(b, a, ppg)
    b, a = butter(4, CFG["lp_art_hz"], btype="low", fs=FS)
    riiv = filtfilt(b, a, ppg)
    n5 = 5 * FS
    tipico = np.median([np.std(riiv[i:i + n5]) for i in range(0, len(riiv) - n5 + 1, n5)])
    filas = []
    for t0 in np.arange(0, len(ppg) / FS - VENTANA_S + 1e-9, PASO_S):
        v = vid[int(t0 * FS_CASE):int((t0 + VENTANA_S) * FS_CASE)]
        if v.min() != v.max() or v[0] not in CATEGORIA:
            continue                                  # solo ventanas dentro de un único vídeo emocional
        i0 = int(round(t0 * FS))
        exc = max(np.std(riiv[i:i + n5]) for i in range(i0, i0 + N - n5 + 1, n5)) / tipico
        filas.append({"Participante": p.stem, "Inicio (s)": t0, "Vídeo": int(v[0]),
                      "Categoría": CATEGORIA[int(v[0])],
                      "RR banda conteo": rr_ref(marcas, t0, t0 + VENTANA_S),
                      "RR banda FFT": rr_fft(rsp125[i0:i0 + N]),
                      "RR PPG dedo": rr_fft(riiv_fft[i0:i0 + N]),
                      "Artefacto": exc > CFG["umbral_art"]})
    return pd.DataFrame(filas)


In [ ]:
# k-3 — Evaluación en todos los participantes
K = pd.concat([evaluar(p) for p in ARCHIVOS], ignore_index=True)
K["Ref fiable"] = (K["RR banda conteo"].between(RPM_MIN, RPM_MAX)
                   & ((K["RR banda conteo"] - K["RR banda FFT"]).abs() <= 2))
K["Válida"] = K["Ref fiable"] & ~K["Artefacto"]
K["Error"] = (K["RR PPG dedo"] - K["RR banda conteo"]).abs()
K.to_csv(BASE_DIR / "results" / "case_rr_dedo_vs_banda.csv", index=False)
print(f"{len(K)} ventanas | referencia fiable {100 * K['Ref fiable'].mean():.0f} % | "
      f"válidas {K['Válida'].sum()} ({100 * K['Válida'].mean():.0f} %)")


In [ ]:
# k-4 — Resultados
V = K[K["Válida"]]
lb = V[V["Categoría"] == "relajante"].groupby("Participante")["RR banda conteo"].mean()

def resumen(g):
    ps = g.groupby("Participante")["Error"].mean()
    return pd.Series({"Participantes": ps.size, "Ventanas": len(g),
                      "MAE mediano por sujeto": ps.median(), "MAE medio": g["Error"].mean(),
                      "% ≤ 2 rpm": 100 * (g["Error"] <= 2).mean(),
                      "RR banda media": g["RR banda conteo"].mean(), "RR PPG media": g["RR PPG dedo"].mean()})

tabla = pd.concat({"Todas": resumen(V), **{c: resumen(g) for c, g in V.groupby("Categoría")}}, axis=1).T
print(tabla.round(2).to_string())
e_triv = (V["Participante"].map(lb) - V["RR banda conteo"]).abs()
print(f"\nLínea base trivial (RR media del participante en vídeos relajantes): MAE medio {e_triv.mean():.2f} rpm")
print("Comparación: BIDMC 1.42 · CapnoBase 0.36 · WESAD muñeca 4.69 (MAE mediano / medio, rpm)")
